# Conditional Mutual Information (CMI) Framework for Comparing Predictive Targets

## 1. Introduction

The goal of this framework is to measure the **unique predictive information** a score $S$ provides about a binary label $Y$, given some side information $Z$. In our application, $S$ represents the predictions (probabilities) from a classification model, and $Z$ can be other model predictions, features, or even alternative target definitions.

Conditional Mutual Information (CMI) is defined as:

$$
I(S;Y \mid Z) = \mathbb{E}\left[ \log \frac{p(Y \mid S,Z)}{p(Y \mid Z)} \right].
$$

This quantifies the reduction in uncertainty about $Y$ when $S$ is known, in addition to knowing $Z$.

## 2. Entropy Reduction Formulation

For binary classification, the CMI can be expressed in terms of the **binary entropy function**:

$$
H_b(p) = -p \log p - (1-p) \log(1-p),
$$

where $p \in (0,1)$ and all logarithms are natural (nats).

Using the definition of conditional entropy, we have:

$$
I(S;Y \mid Z) = H(Y \mid Z) - H(Y \mid S, Z).
$$

For binary $Y$, the conditional entropy terms can be written as expectations of the binary entropy function:

$$
H(Y \mid Z) = \mathbb{E}_Z \big[ H_b(p(Y=1 \mid Z)) \big],
$$

$$
H(Y \mid S, Z) = \mathbb{E}_{S,Z} \big[ H_b(p(Y=1 \mid S,Z)) \big].
$$

Therefore:

$$
I(S;Y \mid Z) = \mathbb{E} \big[ H_b(p(Y=1 \mid Z)) - H_b(p(Y=1 \mid S,Z)) \big].
$$

This is the basis of our estimation procedure.

## 3. Estimation via Cross-Entropy Reduction

We estimate the probabilities $p(Y \mid Z)$ and $p(Y \mid S,Z)$ using probabilistic classifiers.

Let:

* $\hat{p}_0(Z)$ be the estimated $p(Y=1 \mid Z)$
* $\hat{p}_1(S,Z)$ be the estimated $p(Y=1 \mid S,Z)$

The empirical CMI estimate is:

$$
\widehat{I}(S;Y \mid Z) = \frac{1}{n} \sum_{i=1}^n \big[ H_b(\hat{p}_0(Z_i)) - H_b(\hat{p}_1(S_i,Z_i)) \big].
$$

This is equivalent to the average **log-loss gain** from adding $S$ to $Z$ as predictors:

$$
\text{LL gain} = \frac{1}{n} \sum_{i=1}^n \left[ \ell(Y_i, \hat{p}_0(Z_i)) - \ell(Y_i, \hat{p}_1(S_i,Z_i)) \right],
$$

where:

$$
\ell(y, p) = -y \log p - (1-y) \log (1-p)
$$

is the binary cross-entropy loss.

## 4. Cross-Fitting for Unbiasedness

Directly fitting models on the same data used for evaluation can introduce optimistic bias. We address this using **cross-fitting**:

1. Partition the data into $K$ folds.
2. For each fold $f$:

   * Fit $\mathcal{M}_Z$ on $Z_{\text{train}}$ and $Y_{\text{train}}$.
   * Fit $\mathcal{M}_{SZ}$ on $[S_{\text{train}}, Z_{\text{train}}]$ and $Y_{\text{train}}$.
   * Predict $\hat{p}_0(Z_{\text{test}})$ and $\hat{p}_1(S_{\text{test}}, Z_{\text{test}})$.
   * Compute entropy reduction and log-loss gain on the held-out fold.
3. Average across folds.

This ensures that predictions are **out-of-sample** with respect to the auxiliary estimation models.

## 5. Application to Comparing Targets

Suppose we have two targets:

* $Y^{(1)}$ — original (loan-level) target
* $Y^{(2)}$ — modified (customer-level) target

We can train models to produce scores $S^{(1)}$ and $S^{(2)}$ for each.

We then compute:

1. **Self-information:**

$$
I(S^{(1)}; Y^{(1)}) \quad \text{and} \quad I(S^{(2)}; Y^{(2)})
$$

2. **Cross-target CMI:**

$$
I(S^{(1)}; Y^{(2)} \mid Y^{(1)}) \quad \text{and} \quad I(S^{(2)}; Y^{(1)} \mid Y^{(2)})
$$

These measure how much information each score gives about the *other* target that is not already contained in the original target.

3. **Unique score contribution:**

$$
I(S^{(1)}; Y^{(2)} \mid S^{(2)}) \quad \text{and} \quad I(S^{(2)}; Y^{(1)} \mid S^{(1)})
$$

These measure whether one score contains predictive information about the *other* target that is not already captured by the competing score.

## 6. Interpretation

* **High self-information**: Model is predictive for its own target.
* **Low cross-target CMI**: The two targets may encode similar information; one does not add much to predicting the other.
* **Low unique score contribution**: Models are redundant with respect to predicting each other’s targets.
* **High unique score contribution**: One target/model captures aspects of the outcome space that the other misses.

## 7. Units and Scaling

All values are in **nats** (natural log). To convert to **bits**, divide by $\log 2$.

## 8. Caveats

* Accuracy of CMI depends on the calibration of the auxiliary models.
* Cross-fitting mitigates, but does not fully eliminate, bias from finite-sample effects.
* If $S$ is deterministic given $Z$, then $I(S;Y \mid Z) = 0$ by definition.

---

This framework provides a principled, information-theoretic way to compare predictive targets in credit risk or any other binary classification domain, going beyond AUC to measure *unique predictive content*.


# Imports

In [6]:
# -*- coding: utf-8 -*-
"""
Leakage-safe, cross-fitted prototype to compare two binary targets (Y1 vs Y2)
using AUC and Conditional Mutual Information (CMI).

What it does
------------
1) Synthesizes features X and two distinct binary targets Y1, Y2 with different p(Y|X).
2) Trains logistic models and produces OUT-OF-FOLD (OOF) scores for Y1 and Y2 (no leakage).
3) Estimates CMI via cross-fitted entropy reduction:
      I(S;Y|Z) ≈ E[ H_b(p(Y|Z)) - H_b(p(Y|S,Z)) ]
   where S is a score (probability) and Z can be empty, the other label, or the other score.
4) Reports:
   - AUCs for Y1, Y2 (OOF and clean train/test)
   - MI(S1;Y1), MI(S2;Y2)
   - Cross-target info: I(S1;Y2|Y1), I(S2;Y1|Y2)
   - Unique info beyond the other score: I(S1;Y2|S2), I(S2;Y1|S1)

Notes
-----
- For a *final* analysis, increase n_splits and sample size.
- You can swap the auxiliary estimator used in CMI with tree models if relationships are nonlinear.
"""

import numpy as np
from numpy.random import RandomState
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import KFold, train_test_split
from sklearn.base import clone

# Utilities

In [7]:
# ---------------------------
# Utilities
# ---------------------------

def binary_entropy(p, eps=1e-12):
    """
    Compute the binary entropy in natural units (nats).

    The binary entropy function is defined as:

        H_b(p) = - p log p - (1 - p) log (1 - p)

    where:
      - p ∈ [0, 1] is the Bernoulli parameter
      - log is the natural logarithm
      - H_b(p) is measured in nats (not bits)

    This implementation clips `p` to the interval [eps, 1 - eps] to avoid
    numerical issues at the boundaries p = 0 or p = 1.
    """
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def sigmoid(z):
    """
    Numerically stable logistic (sigmoid) function:

        σ(z) = 1 / (1 + exp(-z))

    Maps real-valued inputs to the interval (0, 1), often interpreted
    as probabilities in logistic regression.
    """
    return 1.0 / (1.0 + np.exp(-z))


# ---------------------------
# Data generation
# ---------------------------

def make_synthetic(n=10000, d=20, random_state=123):
    """
    Generate a synthetic dataset with two distinct binary classification
    targets (Y₁, Y₂) defined over the same feature space X, but with
    different conditional probability functions p(Y | X).

    **Feature generation**
    - Create an n × d feature matrix X using sklearn's `make_classification`
      with 10 informative features and 2 redundant features.
    - Class separation and slight label noise are introduced to simulate
      realistic predictive tasks.

    **Target Y₁:**
    A sparse linear logistic model over the first 10 features:

        Let w₁ ∈ ℝᵈ such that w₁[j] = 0 for j ≥ 10.

        p(Y₁ = 1 | X) = σ( X w₁ + ε₁ )

        where:
          - σ(·) is the logistic sigmoid function
          - ε₁ ~ 𝒩(0, 0.25²) is additive Gaussian noise

    **Target Y₂:**
    A different logistic model using a different weight vector and
    an interaction term between the first two features:

        Let w₂ ∈ ℝᵈ with:
            w₂[j] = 0.5 for j < 10
            w₂[j] ~ 𝒩(0, 0.7²) for j ≥ 10

        Define an interaction term:
            int(X) = 0.8 · X₀ · X₁

        Then:

        p(Y₂ = 1 | X) = σ( X w₂ + int(X) + ε₂ )

        where:
          - ε₂ ~ 𝒩(0, 0.25²) is additive Gaussian noise

    **Sampling:**
    For each target, labels are drawn independently:

        Y₁ ~ Bernoulli( p(Y₁ = 1 | X) )
        Y₂ ~ Bernoulli( p(Y₂ = 1 | X) )

    **Returns:**
    - X: np.ndarray of shape (n, d), features
    - y₁: np.ndarray of shape (n,), binary labels for target 1
    - y₂: np.ndarray of shape (n,), binary labels for target 2
    """
    rs = RandomState(random_state)
    X, _ = make_classification(
        n_samples=n, n_features=d, n_informative=10, n_redundant=2,
        n_repeated=0, n_clusters_per_class=2, class_sep=1.25,
        flip_y=0.02, random_state=rs
    )

    # Y1: linear logistic with weights w1 on first 10 features
    w1 = rs.normal(scale=1.0, size=d)
    w1[10:] = 0.0
    logit1 = X @ w1 + rs.normal(scale=0.25, size=n)
    p1 = sigmoid(logit1)
    y1 = (rs.rand(n) < p1).astype(int)

    # Y2: different mapping: different weights + nonlinearity (interaction)
    w2 = rs.normal(scale=1.0, size=d)
    w2[:10] *= 0.5                # damp first 10 a bit
    w2[10:] = rs.normal(scale=0.7, size=d - 10)  # use remaining features too
    inter = 0.8 * X[:, 0] * X[:, 1]              # interaction term for nonlinearity
    logit2 = X @ w2 + inter + rs.normal(scale=0.25, size=n)
    p2 = sigmoid(logit2)
    y2 = (rs.rand(n) < p2).astype(int)

    return X.astype(float), y1, y2


# ---------------------------
# Out-of-fold scoring (no leakage)
# ---------------------------

def oof_scores(model, X, y, n_splits=5, random_state=123):
    """
    Return out-of-fold probability scores for X using classifier `model`.
    Every score is produced by a model that did NOT see that observation's label.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    s = np.empty(len(y), dtype=float)
    for tr, te in kf.split(X):
        m = clone(model)
        m.fit(X[tr], y[tr])
        s[te] = m.predict_proba(X[te])[:, 1]
    return s


# ---------------------------
# Cross-fitted CMI estimation
# ---------------------------

def estimate_cmi_crossfit(y, s, Z,
                          base_estimator=LogisticRegression(max_iter=1000),
                          n_splits=5, random_state=123):
    r"""
    Cross-fitted estimator of the **conditional mutual information** \( I(S;Y \mid Z) \)
    using **expected cross-entropy (log-loss) reduction**.

    **Goal.** Quantify how much extra information a score \(S\) provides about a binary
    label \(Y \in \{0,1\}\) once we already condition on side information \(Z\).
    By the entropy-reduction identity,
    \[
      I(S;Y \mid Z) \;=\; \mathbb{E}\big[\, H_b\!\big(p(Y{=}1 \mid Z)\big)
        \;-\; H_b\!\big(p(Y{=}1 \mid S,Z)\big) \,\big],
    \]
    where \(H_b(p) = -p\log p - (1-p)\log(1-p)\) is the binary entropy (in **nats**).

    **Estimation strategy (cross-fitting).**
    We approximate the two conditional probabilities with probabilistic classifiers:
    \[
      \hat p_0(x) \approx p(Y{=}1 \mid Z), \qquad
      \hat p_1(x) \approx p(Y{=}1 \mid S,Z).
    \]
    To avoid optimistic bias, we use K-fold cross-fitting:

    1) Split indices into `n_splits` folds.  
    2) For each fold \(f\), fit two models on the training part:
       - \( \mathcal{M}_Z \) on \(Z\) to estimate \(p(Y \mid Z)\)
       - \( \mathcal{M}_{SZ} \) on \([S, Z]\) to estimate \(p(Y \mid S, Z)\)
    3) On the held-out fold, compute:
       - Entropy reduction:
         \(\;\Delta H_f = \frac{1}{|f|}\sum_{i\in f}\big[H_b(\hat p_0^{(f)}(Z_i)) - H_b(\hat p_1^{(f)}(S_i,Z_i))\big]\).
       - Log-loss gain:
         \(\;\Delta \mathcal{L}_f = \text{logloss}(Y_f, \hat p_0^{(f)}) - \text{logloss}(Y_f, \hat p_1^{(f)})\).
    4) Average across folds to obtain \(\widehat{I}(S;Y \mid Z)\) and the corresponding mean log-loss gain.

    **Inputs**
    ----------
    y : array-like of shape (n_samples,)
        Binary labels (0/1).

    s : array-like of shape (n_samples,)
        Probabilistic **scores** for the *same* samples (e.g., model posterior \( \hat p(Y{=}1 \mid X) \)).
        Will be reshaped to 2D and concatenated with `Z` for the \(p(Y \mid S,Z)\) model.

    Z : array-like of shape (n_samples, n_features_Z) or (n_samples,)
        Conditioning covariates (features, labels, or other scores). If 1D, it is reshaped to 2D.

    base_estimator : sklearn-like probabilistic classifier, default=LogisticRegression(max_iter=1000)
        Must implement `fit(X, y)` and `predict_proba(X)[:, 1]`. This model is used for both
        \(p(Y \mid Z)\) and \(p(Y \mid S,Z)\). Choose a flexible model if the true relationship is non-linear.

    n_splits : int, default=5
        Number of folds for cross-fitting.

    random_state : int, default=123
        RNG seed for the fold split.

    **Returns**
    -------
    mean_entropy_reduction : float
        \(\widehat{I}(S;Y \mid Z)\) in **nats**, estimated as the average entropy reduction across folds.

    std_entropy_reduction : float
        Standard deviation (across folds) of the entropy-reduction estimates.

    mean_logloss_gain : float
        Average decrease in log-loss when adding \(S\) to \(Z\), i.e.,
        \(\mathbb{E}[\text{LL}(Y \mid Z) - \text{LL}(Y \mid S,Z)]\).
        This is numerically aligned with the entropy reduction but computed via log-loss directly.

    **Interpretation**
    ------------------
    - Larger values indicate that \(S\) provides **unique** predictive information about \(Y\)
      beyond what \(Z\) already explains.
    - A value near 0 suggests \(S\) is (conditionally) redundant given \(Z\).

    **Units**
    --------
    All quantities use natural logarithms; results are in **nats** (not bits).

    **Assumptions & Caveats**
    -------------------------
    - The estimate is only as good as the calibration/fit of the auxiliary models for
      \(p(Y\mid Z)\) and \(p(Y\mid S,Z)\). Severe misspecification can **underestimate** CMI.
    - Cross-fitting reduces overfitting bias; increasing `n_splits` or repeating with multiple seeds
      can improve stability.
    - If `s` is not truly out-of-sample (e.g., produced on the same data used to train the model
      that generated it), residual leakage can inflate the estimate. Prefer OOF scores for `s`.

    """
    y = np.asarray(y).astype(int)
    s = np.asarray(s).reshape(-1, 1)
    Z = np.asarray(Z)
    if Z.ndim == 1:
        Z = Z.reshape(-1, 1)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    ent_diffs, ll_gains = [], []

    for tr, te in kf.split(y):
        m_Z = clone(base_estimator)
        m_SZ = clone(base_estimator)

        m_Z.fit(Z[tr], y[tr])
        m_SZ.fit(np.hstack([s[tr], Z[tr]]), y[tr])

        p0 = m_Z.predict_proba(Z[te])[:, 1]                     # p(Y|Z)
        p1 = m_SZ.predict_proba(np.hstack([s[te], Z[te]]))[:, 1]  # p(Y|S,Z)

        ent_diff = np.mean(binary_entropy(p0) - binary_entropy(p1))
        # log_loss lower is better; gain is LL(Z) - LL(S,Z)
        ll_gain = log_loss(y[te], p0, labels=[0, 1]) - log_loss(y[te], p1, labels=[0, 1])

        ent_diffs.append(ent_diff)
        ll_gains.append(ll_gain)

    return float(np.mean(ent_diffs)), float(np.std(ent_diffs)), float(np.mean(ll_gains))


# ---------------------------
# Evaluation wrapper
# ---------------------------

def evaluate(X, y1, y2,
             clf_proto=LogisticRegression(max_iter=2000),
             aux_estimator=LogisticRegression(max_iter=1000),
             n_splits_oof=5, n_splits_cmi=5, random_state=2024):
    """
    - Builds OOF scores s1, s2 (no leakage)
    - Computes AUCs (OOF and a proper train/test)
    - Estimates MI/CMI quantities with cross-fitting
    """
    # OOF scores for each target
    s1 = oof_scores(clf_proto, X, y1, n_splits=n_splits_oof, random_state=random_state)
    s2 = oof_scores(clf_proto, X, y2, n_splits=n_splits_oof, random_state=random_state)

    # AUCs on a clean train/test split (fit on train only)
    X_tr, X_te, y1_tr, y1_te, y2_tr, y2_te = train_test_split(
        X, y1, y2, test_size=0.3, random_state=random_state, stratify=y1
    )
    clf1 = clone(clf_proto).fit(X_tr, y1_tr)
    clf2 = clone(clf_proto).fit(X_tr, y2_tr)
    auc1 = roc_auc_score(y1_te, clf1.predict_proba(X_te)[:, 1])
    auc2 = roc_auc_score(y2_te, clf2.predict_proba(X_te)[:, 1])

    # OOF AUCs (also valid, but note different variance properties)
    auc1_oof = roc_auc_score(y1, s1)
    auc2_oof = roc_auc_score(y2, s2)

    # MI (no Z): pass a dummy Z of zeros
    Z0 = np.zeros((len(y1), 1))
    mi_s1_y1, mi_s1_y1_sd, _ = estimate_cmi_crossfit(y1, s1, Z0, aux_estimator, n_splits=n_splits_cmi, random_state=random_state)
    mi_s2_y2, mi_s2_y2_sd, _ = estimate_cmi_crossfit(y2, s2, Z0, aux_estimator, n_splits=n_splits_cmi, random_state=random_state)

    # Cross-target (how much S1 tells about Y2 beyond Y1, and vice-versa)
    cmi_s1_y2_given_y1, cmi_sd_12y1, _ = estimate_cmi_crossfit(y2, s1, Z=y1, base_estimator=aux_estimator,
                                                               n_splits=n_splits_cmi, random_state=random_state)
    cmi_s2_y1_given_y2, cmi_sd_21y2, _ = estimate_cmi_crossfit(y1, s2, Z=y2, base_estimator=aux_estimator,
                                                               n_splits=n_splits_cmi, random_state=random_state)

    # Unique info beyond the other score (use the other score as Z)
    cmi_s1_y2_given_s2, _, _ = estimate_cmi_crossfit(y2, s1, Z=s2, base_estimator=aux_estimator,
                                                     n_splits=n_splits_cmi, random_state=random_state)
    cmi_s2_y1_given_s1, _, _ = estimate_cmi_crossfit(y1, s2, Z=s1, base_estimator=aux_estimator,
                                                     n_splits=n_splits_cmi, random_state=random_state)

    return {
        "AUC_holdout": {"Y1": auc1, "Y2": auc2},
        "AUC_OOF": {"Y1": auc1_oof, "Y2": auc2_oof},
        "MI": {
            "I(S1;Y1)": (mi_s1_y1, mi_s1_y1_sd),
            "I(S2;Y2)": (mi_s2_y2, mi_s2_y2_sd),
        },
        "CrossTarget": {
            "I(S1;Y2|Y1)": (cmi_s1_y2_given_y1, cmi_sd_12y1),
            "I(S2;Y1|Y2)": (cmi_s2_y1_given_y2, cmi_sd_21y2),
        },
        "UniqueInfo": {
            "I(S1;Y2|S2)": cmi_s1_y2_given_s2,
            "I(S2;Y1|S1)": cmi_s2_y1_given_s1,
        }
    }



In [8]:

# ---------------------------
# Main
# ---------------------------

if __name__ == "__main__":
    X, y1, y2 = make_synthetic(n=20000, d=30, random_state=7)

    # Try LR for both the base classifier and the auxiliary estimator (fits the synthetic data well).
    base_clf = LogisticRegression(max_iter=2000)
    aux_est  = LogisticRegression(max_iter=1000)

    results = evaluate(
        X, y1, y2,
        clf_proto=base_clf,
        aux_estimator=aux_est,
        n_splits_oof=5,
        n_splits_cmi=5,
        random_state=2024
    )

    # Pretty print
    print("\n=== AUC (holdout) ===")
    for k, v in results["AUC_holdout"].items():
        print(f"{k}: {v:.4f}")

    print("\n=== AUC (OOF) ===")
    for k, v in results["AUC_OOF"].items():
        print(f"{k}: {v:.4f}")

    print("\n=== Mutual Information (nats) ===")
    for k, (m, sd) in results["MI"].items():
        print(f"{k}: {m:.4f} ± {sd:.4f}")

    print("\n=== Cross-Target CMI (nats) ===")
    for k, (m, sd) in results["CrossTarget"].items():
        print(f"{k}: {m:.4f} ± {sd:.4f}")

    print("\n=== Unique Info beyond the other score (nats) ===")
    for k, v in results["UniqueInfo"].items():
        print(f"{k}: {v:.4f}")


=== AUC (holdout) ===
Y1: 0.9534
Y2: 0.9213

=== AUC (OOF) ===
Y1: 0.9553
Y2: 0.9190

=== Mutual Information (nats) ===
I(S1;Y1): 0.4043 ± 0.0046
I(S2;Y2): 0.3117 ± 0.0041

=== Cross-Target CMI (nats) ===
I(S1;Y2|Y1): 0.0003 ± 0.0001
I(S2;Y1|Y2): 0.0004 ± 0.0002

=== Unique Info beyond the other score (nats) ===
I(S1;Y2|S2): 0.0001
I(S2;Y1|S1): 0.0002
